In [4]:
import torch
import torch.nn as nn

In [2]:
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_dim:int
    n_experts:int
    top_k:int
    dropout:float
    batch:int
    seq_len:int
    add_noise:bool
args = ModelArgs(n_dim=32, n_experts=4, top_k=2, dropout=0.1, batch=1, seq_len=5, add_noise=True)
print(args)


ModelArgs(n_dim=32, n_experts=4, top_k=2, dropout=0.1, batch=1, seq_len=5, add_noise=True)


# 实现 router路由

In [7]:
#门控的简单设计
mha_output = torch.randn(args.batch,args.seq_len,args.n_dim)
gate_linear = nn.Linear(args.n_dim, args.n_experts)

logits = gate_linear(mha_output)
print(logits.shape)
print("加噪前：", logits)

if args.add_noise:
    #normal(logits) * softplus(w_noise * x_input)
    noise_linear = nn.Linear(args.n_dim, args.n_experts)
    noise_logits = noise_linear(mha_output)
    #生成与 logits（门控 logit）同形状的标准正态随机噪声
    noise = torch.randn_like(logits) * nn.functional.softplus(noise_logits)
    logits = logits + noise
    print("加噪后：", logits)

torch.Size([1, 5, 4])
加噪前： tensor([[[-0.5023,  0.2996,  0.5293,  1.2390],
         [ 0.5768, -0.9195, -1.1175, -0.3952],
         [ 0.2781,  0.0098,  0.0777,  0.5496],
         [-0.0989,  0.8893,  0.1649,  0.2634],
         [ 0.9121, -0.1760, -0.3334, -0.6584]]], grad_fn=<ViewBackward0>)
加噪后： tensor([[[-0.1423,  0.0722,  0.9155, -0.4370],
         [ 1.8146, -0.6096, -1.4144, -1.1225],
         [ 0.4633,  0.0839,  0.2063, -0.0066],
         [ 0.6498,  0.8908,  0.2118,  0.8213],
         [ 0.7542, -0.3088, -0.5667, -1.1209]]], grad_fn=<AddBackward0>)


# TOP-K选择

In [8]:
top_k_logits,top_k_indices = logits.topk(args.top_k,dim=-1)
top_k_logits, top_k_indices

(tensor([[[ 0.9155,  0.0722],
          [ 1.8146, -0.6096],
          [ 0.4633,  0.2063],
          [ 0.8908,  0.8213],
          [ 0.7542, -0.3088]]], grad_fn=<TopkBackward0>),
 tensor([[[2, 1],
          [0, 1],
          [0, 2],
          [1, 3],
          [0, 1]]]))

# 加softmax

In [11]:
infs = torch.full_like(logits,float("-inf"))
print(infs)
sparse_logits = infs.scatter(-1, top_k_indices, top_k_logits)
sparse_logits

tensor([[[-inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf],
         [-inf, -inf, -inf, -inf]]])


tensor([[[   -inf,  0.0722,  0.9155,    -inf],
         [ 1.8146, -0.6096,    -inf,    -inf],
         [ 0.4633,    -inf,  0.2063,    -inf],
         [   -inf,  0.8908,    -inf,  0.8213],
         [ 0.7542, -0.3088,    -inf,    -inf]]], grad_fn=<ScatterBackward0>)

# 门控输出

In [12]:
gating_output = nn.functional.softmax(sparse_logits,dim=-1)
gating_output

tensor([[[0.0000, 0.3008, 0.6992, 0.0000],
         [0.9187, 0.0813, 0.0000, 0.0000],
         [0.5639, 0.0000, 0.4361, 0.0000],
         [0.0000, 0.5174, 0.0000, 0.4826],
         [0.7433, 0.2567, 0.0000, 0.0000]]], grad_fn=<SoftmaxBackward0>)

# 实现Top_K的路由网络

In [14]:
class TopkRouter(nn.Module):
    '''定义一个TOP-K的门控单元
    '''
    def __init__(self,args):
        super().__init__()
        self.top_k = args.top_k

         # top-k的学习层(门控的参数）
        self.top_k_linear = nn.Linear(args.n_dim, args.n_experts)

         # 噪声的学习层，定义一个噪声可学习的参数
        self.noise_linear = nn.Linear(args.n_dim, args.n_experts)

    def forward(self,mha_output):
        #logits
        logits = self.top_k_linear(mha_output)

        #noise_logits
        noise_logits = self.noise_linear(mha_output)

        #对logits 添加高斯噪声
        noise = torch.randn_like(logits) * nn.functional.softplus(noise_logits)
        logits = logits + noise

        #选择top_k
        top_k_logits,top_k_indices = logits.topk(self.top_k,dim=-1)

        #构造sparse的logits
        infs = torch.full_like(logits,float("-inf"))
        sparse_logits = infs.scatter(-1,top_k_indices, top_k_logits)

        #计算路由输出
        gating_output = nn.functional.softmax(sparse_logits,dim=-1)
        return gating_output,top_k_indices

In [15]:
# 固定随机种子
torch.manual_seed(1)

mha_output = torch.randn(args.batch, args.seq_len, args.n_dim)

noisy_top_k_router = TopkRouter(args)
gating_output, top_k_indices = noisy_top_k_router(mha_output)

gating_output.shape, gating_output, top_k_indices

(torch.Size([1, 5, 4]),
 tensor([[[0.6560, 0.3440, 0.0000, 0.0000],
          [0.0000, 0.2567, 0.0000, 0.7433],
          [0.0000, 0.6651, 0.0000, 0.3349],
          [0.0000, 0.8481, 0.1519, 0.0000],
          [0.0000, 0.2715, 0.0000, 0.7285]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[0, 1],
          [3, 1],
          [1, 3],
          [1, 2],
          [3, 1]]]))

# 实现专家网络

In [16]:
class Expert(nn.Module):
    '''专家网络的实现，用一个标准的FFN层来作为expert
    '''

    def __init__(self,args):
        super().__init__()

        self.ffn = nn.Sequential(
            nn.Linear(args.n_dim,4*args.n_dim),
            nn.ReLU(),
            nn.Linear(4*args.n_dim,args.n_dim),
            nn.Dropout(args.dropout)
        )

    def forward(self,x):
        return self.ffn(x)

# 定义sparse MOE架构网络

In [21]:
class SparseMOE(nn.Module):
    '''定义完整的MOE网络'''

    def __init__(self,args):
        super().__init__()

        self.router = TopkRouter(args)
        self.experts = nn.ModuleList([Expert(args) for _ in range(args.n_experts)]) #创建 N 个专家网络的列表。
        self.top_k = args.top_k

    def forward(self,x):
        #门控的路由输出
        gating_output,indices = self.router(x)
        print("select experts: ", indices)

        #定义输出
        final_output = torch.zeros_like(x)

        '''路由器是基于单个 Token 做决策的，
        而不是基于整个句子或批次。因此，我们需要打破 batch（批次）和 seq_len（序列长度）的界限，
        将所有 Token 视为一个巨大的“Token 池”'''
        #Reshape可以让X去做batch的处理
        # [b,l,d] => [b*l,d]
        flat_x = x.view(-1, x.size(-1))
        # [b,l,n] => [b*l,n]
        flat_gating_output = gating_output.view(-1, gating_output.size(-1))

        #并行处理每一个专家
        for i,expert in enumerate(self.experts):
            expert_mask = (indices == i).any(dim=-1)#[batch_size, seq_len]
            flat_mask = expert_mask.view(-1)#[batch_size*seq_len]  摊平

            #如果专家i被选中了
            if flat_mask.any():
                #[select_k, d]，只取该专家被选中token的输入
                expert_input = flat_x[flat_mask] #高级索引，True的位置保留

                # 实际上我们是先算router，再去计算专家输出，没有被选中的专家，不参与计算
                # [select_k, d]
                expert_output = expert(expert_input)

                #专家输出加权求和, 只取被选中专家的得分
                # [select_k] => [select_k, 1]
                gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1)
                # [select_k，1] => [select_k,n]
                weighted_output = expert_output * gating_scores

                #更新一下输出,对[batch_size, seq_len]选中的部分也就是为True的部分[select_k,n]，对应的添加上weighted_output
                # [b, n, d]
                final_output[expert_mask] += weighted_output
        return final_output

In [23]:
torch.manual_seed(1)

mha_output = torch.randn(args.batch, args.seq_len, args.n_dim)
sparse_moe = SparseMOE(args)
final_output = sparse_moe(mha_output)
print("input shape: ", mha_output.shape)
print("output shape: ", final_output.shape)
print(final_output)

select experts:  tensor([[[0, 1],
         [1, 3],
         [1, 2],
         [1, 2],
         [1, 0]]])
input shape:  torch.Size([1, 5, 32])
output shape:  torch.Size([1, 5, 32])
tensor([[[ 0.2008,  0.2713, -0.0292, -0.1669, -0.1428,  0.2245,  0.0407,
           0.2055,  0.3688, -0.0387,  0.1589,  0.0641, -0.0249, -0.1521,
           0.1070,  0.1908,  0.0744,  0.1325,  0.1821, -0.1476,  0.2865,
          -0.3378,  0.2426, -0.1132, -0.1253, -0.1436, -0.5903,  0.1164,
          -0.1296,  0.0697,  0.0508,  0.3190],
         [ 0.2387, -0.0856, -0.1255,  0.0899, -0.3855, -0.0313, -0.0663,
           0.1403, -0.0522, -0.1801,  0.2527,  0.1311, -0.1611,  0.0424,
          -0.2385,  0.0196,  0.2778,  0.2567,  0.1813,  0.2325,  0.1197,
           0.0092,  0.0814,  0.0889, -0.1320,  0.4106, -0.2029, -0.0709,
           0.0000, -0.2215, -0.1610,  0.0561],
         [-0.0162, -0.0774, -0.1275,  0.1245,  0.2023, -0.3166,  0.1053,
          -0.0345,  0.0032,  0.0771,  0.0740,  0.1062, -0.2965, -0.111

# DeepSeek MOE

In [33]:
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_dim:int
    n_experts:int
    n_shared_experts:int  #增加一个共享的专家数
    top_k:int
    dropout:float
    batch:int
    seq_len:int
    add_noise:bool
args = ModelArgs(n_dim=32, n_experts=4, n_shared_experts=1,top_k=2, dropout=0.1, batch=1, seq_len=5, add_noise=True)
print(args)

ModelArgs(n_dim=32, n_experts=4, n_shared_experts=1, top_k=2, dropout=0.1, batch=1, seq_len=5, add_noise=True)


# 重新定义 Expert

In [34]:
class DeepSeekExpert(nn.Module):
    def __init__(self,args,bias=False):
        super().__init__()

        self.gate_proj = nn.Linear(args.n_dim,4*args.n_dim,bias=False)
        self.up_proj = nn.Linear(args.n_dim, 4*args.n_dim, bias=False)
        self.down_proj = nn.Linear(4*args.n_dim, args.n_dim, bias=False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)

        return self.down_proj(nn.functional.silu(gate) * up)

# 实现DeepSeekMOE

In [37]:
class DeepSeekMOE(nn.Module):
    '''定义完整的 DeepSeek MOE网络
     '''

    def __init__(self,args):
        super().__init__()

        self.router = TopkRouter(args)
        self.routed_experts = nn.ModuleList([DeepSeekExpert(args) for _ in range(args.n_experts)])
        self.shared_experts = nn.ModuleList([DeepSeekExpert(args) for _ in range(args.n_shared_experts)])
        self.top_k = args.top_k

    def forward(self,x):
        #门控的路由输出
        gating_output,indices = self.router(x)
        print("select experts: ", indices)

        #定义输出
        final_output = torch.zeros_like(x)

        '''路由器是基于单个 Token 做决策的，
        而不是基于整个句子或批次。因此，我们需要打破 batch（批次）和 seq_len（序列长度）的界限，
        将所有 Token 视为一个巨大的“Token 池”'''
        #Reshape可以让X去做batch的处理
        # [b,l,d] => [b*l,d]
        flat_x = x.view(-1, x.size(-1))
        # [b,l,n] => [b*l,n]
        flat_gating_output = gating_output.view(-1, gating_output.size(-1))

        #并行处理每一个路由专家
        for i,expert in enumerate(self.routed_experts):
            expert_mask = (indices == i).any(dim=-1)#[batch_size, seq_len]
            flat_mask = expert_mask.view(-1)#[batch_size*seq_len]  摊平

            #如果专家i被选中了
            if flat_mask.any():
                #[select_k, d]，只取该专家被选中token的输入
                expert_input = flat_x[flat_mask] #高级索引，True的位置保留

                # 实际上我们是先算router，再去计算专家输出，没有被选中的专家，不参与计算
                # [select_k, d]
                expert_output = expert(expert_input)

                #专家输出加权求和, 只取被选中专家的得分
                # [select_k] => [select_k, 1]
                gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1)
                # [select_k，1] => [select_k,n]
                weighted_output = expert_output * gating_scores

                #更新一下输出
                # [b, n, d]
                final_output[expert_mask] += weighted_output.squeeze(1)
                        # 并行处理每一个shared专家
        for i, expert in enumerate(self.shared_experts):
            # [b, n, d]
            final_output += expert(flat_x)
        return final_output, indices

In [38]:
torch.manual_seed(1)

mha_output = torch.randn(args.batch, args.seq_len, args.n_dim)
deepseek_moe = DeepSeekMOE(args)
final_output, _ = deepseek_moe(mha_output)
print("input shape: ", mha_output.shape)
print("output shape: ", final_output.shape)
print(final_output)


select experts:  tensor([[[0, 2],
         [1, 3],
         [3, 2],
         [1, 3],
         [2, 0]]])
input shape:  torch.Size([1, 5, 32])
output shape:  torch.Size([1, 5, 32])
tensor([[[-0.1060, -0.0898,  0.0438, -0.1815,  0.0314,  0.0218, -0.1636,
          -0.0152,  0.0162, -0.4288,  0.0639,  0.1033, -0.0939, -0.0976,
          -0.0170, -0.0946,  0.0242,  0.1025,  0.0774,  0.2320,  0.1367,
           0.1103,  0.1187, -0.1827,  0.0992, -0.2188,  0.1022, -0.1033,
          -0.0952,  0.1780,  0.1084, -0.2307],
         [-0.0434,  0.0534, -0.1539,  0.0208, -0.1863, -0.2880, -0.3897,
          -0.0597,  0.0592,  0.0157,  0.0052, -0.1533,  0.0912,  0.0223,
          -0.1085,  0.0654,  0.1534, -0.1833, -0.0679, -0.1376, -0.0179,
          -0.0416, -0.1209,  0.0731, -0.0820,  0.2760,  0.1484, -0.3519,
           0.1481, -0.0183,  0.1958,  0.1356],
         [ 0.0995, -0.0097,  0.0273,  0.0557,  0.0641, -0.0432, -0.0836,
          -0.0104,  0.0182,  0.0076,  0.0323,  0.1624,  0.0922, -0.122